# Κεφάλαιο 11: Φτιάξε το Transformer

> «Είμαστε αυτό που κάνουμε επανειλημμένα. Η αριστεία, λοιπόν, δεν είναι πράξη, αλλά συνήθεια.»
> — **Αριστοτέλης**, φιλόσοφος

---

## Τι θα μάθεις

- Πώς να στοιβάξεις μπλοκ Transformer για να φτιάξεις ένα πλήρες γλωσσικό μοντέλο
- Ποιο πρότυπο διαμόρφωσης διευκολύνει τον πειραματισμό με μοντέλα
- Τι κάνει η κεφαλή γλωσσικής μοντελοποίησης και γιατί τη χρειαζόμαστε
- Πώς η κοινή χρήση βαρών στο GPT-2 Small εξοικονομεί 38.597.376 παραμέτρους
- Ποιους βασικούς ελέγχους να κάνεις για να βεβαιωθείς ότι το μοντέλο σου λειτουργεί σωστά πριν από την εκπαίδευση
- Πώς να φορτώσεις τα προεκπαιδευμένα βάρη του GPT-2 στη δική σου αρχιτεκτονική

---

## Προετοιμασία

Άνοιξε το notebook [απευθείας στο Colab](https://colab.research.google.com/github/FirstLLM/code/blob/main/notebooks/el/ch11.ipynb) ή κατέβασε το [αρχείο για το Jupyter](https://github.com/FirstLLM/code/blob/main/notebooks/el/ch11.ipynb).

Πρώτα, εγκατάστησε τα απαραίτητα πακέτα:

In [ ]:
# Εγκατάστησε τα απαραίτητα πακέτα
!pip install -q torch==2.15.0 transformers==5.23.0 matplotlib==3.10.7

In [ ]:
# ===== ΕΙΣΑΓΩΓΕΣ =====
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from transformers import AutoTokenizer

# Όρισε τη συσκευή
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Συσκευή σε χρήση: {device}")

## 1. Δομικά στοιχεία από το Κεφάλαιο 10

Πρώτα, εισήγαγε τα δομικά στοιχεία που φτιάξαμε στο Κεφάλαιο 10: `MultiHeadAttention`, `FeedForward` και `TransformerBlock`.

In [ ]:
# ===== ΠΡΟΣΟΧΗ ΠΟΛΛΑΠΛΩΝ ΚΕΦΑΛΩΝ (από το Κεφάλαιο 10) =====
class MultiHeadAttention(nn.Module):
    """
    Αποδοτική προσοχή πολλαπλών κεφαλών (επεξεργάζεται όλες τις κεφαλές σε μία παρτίδα).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "Το d_model πρέπει να διαιρείται ακριβώς με το num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64

        # Ενιαία προβολή QKV (3 φορές πιο αποδοτική από τις χωριστές προβολές!)
        # Γιατί 3 * d_model; Επειδή προβάλλουμε την είσοδο στα Q, K, V ταυτόχρονα
        # Το GPT-2 χρησιμοποιεί bias=True, οπότε μπορούμε να φορτώσουμε και τους προεκπαιδευμένους όρους μετατόπισης.
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # Προβολή εξόδου
        self.out_proj = nn.Linear(d_model, d_model)

        # Dropout στα βάρη προσοχής
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        # ===== Βήμα 1: Κάνε προβολή στα Q, K, V (όλα ταυτόχρονα!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)

        # ===== Βήμα 2: Χώρισε το αποτέλεσμα σε Q, K, V και προσάρμοσε το σχήμα για τις πολλαπλές κεφαλές =====
        # Άλλαξε το σχήμα σε (batch, seq, 3, num_heads, d_head)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)

        # Αναδιάταξε τους άξονες σε (3, batch, num_heads, seq, d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Χώρισε τον τανυστή σε Q, K, V: καθένα έχει σχήμα (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # ===== Βήμα 3: Προσοχή με κλιμακωτό εσωτερικό γινόμενο (υπολογισμός σε μία παρτίδα για όλες τις κεφαλές) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)

        # Εφάρμοσε την αιτιακή μάσκα, αν έχει δοθεί
        if mask is not None:
            # Επέκτεινε τη μάσκα για τις κεφαλές: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)

        # Σταθμισμένο άθροισμα των τιμών
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)

        # ===== Βήμα 4: Συνένωσε τις κεφαλές =====
        # Αντάλλαξε τους άξονες ώστε το σχήμα να γίνει (batch, seq, num_heads, d_head)
        attn_output = attn_output.transpose(1, 2)

        # Άλλαξε το σχήμα σε (batch, seq, d_model), ώστε να συνενώσεις τις κεφαλές
        attn_output = attn_output.reshape(batch, seq, d_model)

        # ===== Βήμα 5: Τελική προβολή =====
        output = self.out_proj(attn_output)

        return output, attn_weights




In [ ]:
# ===== ΔΙΚΤΥΟ ΠΡΟΣΘΙΑΣ ΤΡΟΦΟΔΟΤΗΣΗΣ (από το Κεφάλαιο 10) =====
class FeedForward(nn.Module):
    """
    Δίκτυο πρόσθιας τροφοδότησης ανά θέση.
    Εφαρμόζεται σε κάθε θέση ανεξάρτητα (με τα ίδια βάρη για όλες τις θέσεις).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Ορίσματα:
            d_model: Διάσταση του μοντέλου (768 για το μικρό GPT-2)
            d_ff: Διάσταση του δικτύου πρόσθιας τροφοδότησης (συνήθως 4 * d_model = 3072)
            dropout: Πιθανότητα απενεργοποίησης
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff), αύξηση της διάστασης
        x = F.gelu(x, approximate="tanh")  # Μη γραμμικότητα (παραλλαγή της GELU που χρησιμοποιεί το GPT-2)
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model), προβολή στην αρχική διάσταση
        return x



print("Ορίστηκε το FeedForward!")

In [ ]:
# ===== ΜΠΛΟΚ Transformer (από το Κεφάλαιο 10) =====
class TransformerBlock(nn.Module):
    """
    Πλήρες μπλοκ Transformer με προσοχή πολλαπλών κεφαλών, δίκτυο πρόσθιας τροφοδότησης,
    υπολειμματικές συνδέσεις και κανονικοποίηση επιπέδου (πριν από κάθε υποεπίπεδο, όπως στο GPT-2).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Ορίσματα:
            d_model: Διάσταση του μοντέλου (768)
            num_heads: Αριθμός κεφαλών προσοχής (12)
            d_ff: Διάσταση του δικτύου πρόσθιας τροφοδότησης (3072 = 4 * d_model)
            dropout: Πιθανότητα απενεργοποίησης
        """
        super().__init__()

        # Κανονικοποίηση επιπέδου (πριν από κάθε υποεπίπεδο)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Προσοχή πολλαπλών κεφαλών
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)

        # Δίκτυο πρόσθιας τροφοδότησης
        self.ffn = FeedForward(d_model, d_ff, dropout)

        # Dropout (εφαρμόζεται μετά από κάθε υποεπίπεδο)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Ορίσματα:
            x: Διανυσματικές αναπαραστάσεις εισόδου (batch, seq, d_model)
            mask: Αιτιακή μάσκα (seq, seq) ή (1, 1, seq, seq)

        Επιστρέφει:
            x: Διανυσματικές αναπαραστάσεις εξόδου (batch, seq, d_model)
            attn_weights: Βάρη προσοχής (batch, heads, seq, seq)
        """
        # ===== Προσοχή πολλαπλών κεφαλών με υπολειμματική σύνδεση =====
        # Προκανονικοποίηση: Κανονικοποίησε την είσοδο ΠΡΙΝ από την προσοχή
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Υπολειμματική σύνδεση

        # ===== Δίκτυο πρόσθιας τροφοδότησης με υπολειμματική σύνδεση =====
        # Προκανονικοποίηση: Κανονικοποίησε την είσοδο ΠΡΙΝ από το δίκτυο πρόσθιας τροφοδότησης
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Υπολειμματική σύνδεση

        return x, attn_weights




## 2. Ρύθμιση του μοντέλου

Δημιούργησε μια κλάση δεδομένων για να συγκεντρώσεις όλες τις υπερπαραμέτρους του μοντέλου.

In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """Ρυθμίσεις για το μοντέλο MiniGPT."""
    vocab_size: int = 50257      # Μέγεθος του λεξιλογίου του GPT-2
    max_seq_len: int = 1024      # Μέγιστο μήκος συμφραζομένων
    embed_dim: int = 768         # Διάσταση της διανυσματικής αναπαράστασης (d_model)
    num_heads: int = 12          # Αριθμός κεφαλών προσοχής
    num_layers: int = 12         # Αριθμός μπλοκ Transformer
    d_ff: int = 3072             # Διάσταση του κρυφού επιπέδου του δικτύου πρόσθιας τροφοδότησης (4 * embed_dim)
    dropout: float = 0.1         # Πιθανότητα dropout

    def __post_init__(self):
        """Έλεγξε την εγκυρότητα των ρυθμίσεων μετά την αρχικοποίηση."""
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}) πρέπει να διαιρείται ακριβώς με " \
            f"num_heads ({self.num_heads})"


## 3. Το πλήρες μοντέλο MiniGPT

Τώρα φτιάξε το πλήρες μοντέλο στοιβάζοντας μπλοκ Transformer και προσθέτοντας την κεφαλή γλωσσικής μοντελοποίησης με κοινή χρήση βαρών.

In [ ]:
class MiniGPT(nn.Module):
    """
    Ένα λιτό γλωσσικό μοντέλο τύπου GPT.
    Συνδυάζει διανυσματικές αναπαραστάσεις, μπλοκ Transformer και μια κεφαλή γλωσσικής μοντελοποίησης.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # ===== Διανυσματικές αναπαραστάσεις token και θέσεων (Κεφάλαιο 9) =====
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # ===== Μπλοκ Transformer (Κεφάλαιο 10) =====
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # ===== Τελικό LayerNorm =====
        self.ln_f = nn.LayerNorm(config.embed_dim)

        # ===== Κεφαλή γλωσσικής μοντελοποίησης =====
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # ===== Κοινή χρήση βαρών =====
        # Χρησιμοποίησε τα ίδια βάρη για τη διανυσματική αναπαράσταση και την προβολή εξόδου
        self.lm_head.weight = self.token_embed.weight

        # Αρχικοποίησε τα βάρη
        self._init_weights()

    def _init_weights(self):
        """Αρχικοποίησε τα βάρη με μικρές τυχαίες τιμές."""
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        """
        Πρόσθιο πέρασμα από το μοντέλο.

        Ορίσματα:
            token_ids: Αναγνωριστικά των token εισόδου (batch, seq)
            return_attention: Αν θα επιστραφούν τα βάρη προσοχής

        Επιστρέφει:
            logits: Βαθμολογίες για τα token του λεξιλογίου (batch, seq, vocab_size)
            attention_weights: Προαιρετική λίστα με τα βάρη προσοχής ανά μπλοκ
        """
        batch, seq = token_ids.shape
        device = token_ids.device

        # ===== Διανυσματικές αναπαραστάσεις =====
        # Διανυσματικές αναπαραστάσεις token
        tok_emb = self.token_embed(token_ids)  # (batch, seq, embed_dim)

        # Διανυσματικές αναπαραστάσεις θέσεων
        positions = torch.arange(seq, device=device)  # (seq,)
        pos_emb = self.pos_embed(positions)  # (seq, embed_dim)

        # Συνδύασε τις αναπαραστάσεις και εφάρμοσε dropout
        x = self.dropout(tok_emb + pos_emb)  # (batch, seq, embed_dim)

        # ===== Δημιουργία αιτιακής μάσκας =====
        # Η αιτιακή μάσκα επιτρέπει προσοχή στην τρέχουσα και στις προηγούμενες θέσεις.
        mask = torch.tril(torch.ones(seq, seq, device=device))

        # ===== Διέλευση από τα μπλοκ Transformer =====
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        # ===== Τελικό LayerNorm =====
        x = self.ln_f(x)  # (batch, seq, embed_dim)

        # ===== Κεφαλή γλωσσικής μοντελοποίησης =====
        logits = self.lm_head(x)  # (batch, seq, vocab_size)

        if return_attention:
            return logits, attention_weights
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        Παρήγαγε token ως συνέχεια της ακολουθίας input_ids (η προτροπή περιλαμβάνεται στην έξοδο).

        Ορίσματα:
            input_ids: Αναγνωριστικά των token της προτροπής (batch, seq)
            max_new_tokens: Πόσα νέα token θα παραχθούν
            temperature: Θερμοκρασία δειγματοληψίας (χρησιμοποιείται μόνο όταν do_sample=True)
            do_sample: Δειγματοληψία από την κατανομή· False = άπληστη αποκωδικοποίηση με argmax
            eos_token_id: Σταμάτησε νωρίτερα αν κάθε ακολουθία παραγάγει αυτό το token

        Επιστρέφει:
            Αναγνωριστικά των token (batch, seq + new tokens)
        """
        self.eval()

        # Οι διανυσματικές αναπαραστάσεις θέσεων καλύπτουν μόνο max_seq_len θέσεις
        limit = self.config.max_seq_len
        if input_ids.shape[1] + max_new_tokens > limit:
            raise ValueError(
                f"Η προτροπή ({input_ids.shape[1]} token) μαζί με τα max_new_tokens "
                f"({max_new_tokens}) ξεπερνά το όριο των {limit} token "
                "του μοντέλου. Συντόμευσε την προτροπή ή μείωσε το max_new_tokens.")
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids


In [ ]:
# Γρήγορη δοκιμή με ρυθμίσεις για πολύ μικρό μοντέλο
tiny_config = GPTConfig(vocab_size=500, max_seq_len=16, embed_dim=64,
                        num_heads=2, num_layers=2, d_ff=256, dropout=0.0)
model = MiniGPT(tiny_config)

# Έλεγξε το πρόσθιο πέρασμα
test_tokens = torch.randint(0, tiny_config.vocab_size, (2, 16))
logits = model(test_tokens)

print(f"Σχήμα εισόδου: {test_tokens.shape}")
print(f"Σχήμα εξόδου: {logits.shape}")
print(f"\nΑναμενόμενο: (2, 16, {tiny_config.vocab_size})")

## 4. Βασικοί έλεγχοι ορθότητας

Αρχικά, εκτέλεσε πέντε δοκιμές για να ελέγξεις πώς συνδέονται τα μέρη του μοντέλου. Αργότερα, μια έκτη δοκιμή συγκρίνει τα logits μετά τη φόρτωση των βαρών του GPT-2.

In [ ]:
# ===== ΔΟΚΙΜΗ 1: Επαλήθευση σχημάτων =====

def test_forward_shapes(config):
    """Επαλήθευσε τα σχήματα των εξόδων του μοντέλου."""
    model = MiniGPT(config)
    
    batch_size, seq_len = 2, 16
    token_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))
    
    logits = model(token_ids)
    
    expected_shape = (batch_size, seq_len, config.vocab_size)
    assert logits.shape == expected_shape, \
        f"Αναμενόμενο σχήμα: {expected_shape}, πραγματικό: {logits.shape}"
    
    print(f"Ο έλεγχος σχήματος πέτυχε: {logits.shape}")

test_forward_shapes(tiny_config)

In [ ]:
# ===== ΔΟΚΙΜΗ 2: Πλήθος παραμέτρων =====
def count_parameters(model):
    """Μέτρησε το σύνολο των εκπαιδεύσιμων παραμέτρων."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def test_parameter_count(config):
    """Βεβαιώσου ότι το πλήθος των παραμέτρων είναι εύλογο."""
    model = MiniGPT(config)
    total = count_parameters(model)

    # GPT-2 Small: 124.439.808 παράμετροι συνολικά· ο κοινός πίνακας εισόδου και εξόδου μετράει μία φορά.
    print(f"Σύνολο παραμέτρων: {total:,}")

    # Ακριβής κατανομή (μαζί με τους όρους μετατόπισης)
    embed_params = config.vocab_size * config.embed_dim  # Διανυσματικές αναπαραστάσεις των token
    pos_params = config.max_seq_len * config.embed_dim   # Διανυσματικές αναπαραστάσεις θέσεων
    d = config.embed_dim
    block_params = config.num_layers * (
        4 * d * d + 4 * d +                     # Προσοχή: QKV + έξοδος, με όρους μετατόπισης
        2 * d * config.d_ff + config.d_ff + d + # Δίκτυο πρόσθιας τροφοδότησης: fc1 + fc2, με όρους μετατόπισης
        4 * d                                   # Επίπεδα LayerNorm
    )
    ln_f_params = 2 * d                          # Τελικό LayerNorm
    # Σημείωση: Υπάρχει κοινή χρήση βαρών μεταξύ lm_head και token_embed, οπότε οι παράμετροι του lm_head δεν μετρώνται χωριστά

    expected = embed_params + pos_params + block_params + ln_f_params
    print(f"Αναμενόμενο: {expected:,}")
    assert total == expected, f"Ασυμφωνία στο πλήθος παραμέτρων: {total:,} αντί για {expected:,}"


# Δοκιμή με ρυθμίσεις για πολύ μικρό μοντέλο
print("Πολύ μικρό μοντέλο:")
test_parameter_count(tiny_config)

# Δοκιμή με τις ρυθμίσεις του πλήρους GPT-2
print("\nGPT-2 Small:")
test_parameter_count(GPTConfig())

In [ ]:
# ===== ΔΟΚΙΜΗ 3: Αιτιακή μάσκα =====

def test_causal_masking(config):
    """Βεβαιώσου ότι το μοντέλο δεν μπορεί να δει μελλοντικά token."""
    model = MiniGPT(config)
    model.eval()

    # Ίδιο πρόθεμα, διαφορετικό επίθημα
    tokens_a = torch.tensor([[100, 200, 300, 400]])
    tokens_b = torch.tensor([[100, 200, 300, 401]])  # Εντός του πολύ μικρού λεξιλογίου

    with torch.no_grad():
        logits_a = model(tokens_a)
        logits_b = model(tokens_b)

    # Τα logits στις θέσεις 0, 1, 2 πρέπει να είναι ΠΑΝΟΜΟΙΟΤΥΠΑ
    for pos in range(3):
        assert torch.allclose(logits_a[0, pos], logits_b[0, pos], atol=1e-5), \
            f"Τα logits διαφέρουν στη θέση {pos}!"

    # Τα logits στη θέση 3 ΠΡΕΠΕΙ να διαφέρουν
    assert not torch.allclose(logits_a[0, 3], logits_b[0, 3], atol=1e-5), \
        "Τα logits στη θέση 3 είναι ίδια παρά τη διαφορετική είσοδο!"

    print("Η δοκιμή της αιτιακής μάσκας ΠΕΤΥΧΕ!")

test_causal_masking(tiny_config)

In [ ]:
# ===== ΔΟΚΙΜΗ 4: Κοινή χρήση βαρών =====

def test_weight_tying(config):
    """Επαλήθευσε ότι η διανυσματική αναπαράσταση και το lm_head μοιράζονται τα ίδια βάρη."""
    model = MiniGPT(config)

    # Πρέπει να είναι ο ΙΔΙΟΣ τανυστής
    assert model.lm_head.weight is model.token_embed.weight, \
        "Η κοινή χρήση βαρών απέτυχε: οι τανυστές είναι διαφορετικοί!"

    # Τροποποίησε τον έναν και έλεγξε ότι αλλάζει και ο άλλος
    with torch.no_grad():
        original = model.token_embed.weight[0, 0].item()
        model.token_embed.weight[0, 0] = 999.0
        
        assert model.lm_head.weight[0, 0].item() == 999.0, \
            "Η κοινή χρήση βαρών απέτυχε: οι αλλαγές δεν μεταδίδονται!"
        
        model.token_embed.weight[0, 0] = original

    print("Η δοκιμή κοινής χρήσης βαρών ΠΕΤΥΧΕ!")

test_weight_tying(tiny_config)

In [ ]:
# ===== ΔΟΚΙΜΗ 5: Ροή κλίσεων =====

def test_gradient_flow(config):
    """Επαλήθευσε ότι οι κλίσεις φτάνουν σε όλες τις παραμέτρους."""
    model = MiniGPT(config)

    # Πρόσθιο πέρασμα
    tokens = torch.randint(0, config.vocab_size, (1, 8))
    logits = model(tokens)

    # Οπίσθια διέλευση
    loss = logits.sum()
    loss.backward()

    # Έλεγξε ότι όλες οι παράμετροι έχουν κλίσεις
    for name, param in model.named_parameters():
        assert param.grad is not None, f"Δεν υπάρχει κλίση για την παράμετρο {name}"

    print("Η δοκιμή ροής κλίσεων ΠΕΤΥΧΕ!")

test_gradient_flow(tiny_config)

## 5. Το πρώτο σου πρόσθιο πέρασμα

Ας δώσουμε πραγματικό κείμενο στο μοντέλο μας για να δούμε τι παράγει με τυχαία βάρη. Οι προτροπές παραμένουν στα αγγλικά, ώστε ο tokenizer του GPT-2 να δέχεται ακριβώς την ίδια είσοδο όπως στο αρχικό παράδειγμα.

In [ ]:
# Φόρτωσε τον tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Δημιούργησε ένα μικρό μοντέλο
config = GPTConfig(num_layers=2, embed_dim=256, num_heads=4, d_ff=1024)
model = MiniGPT(config)
model.eval()

# Χώρισε μια προτροπή σε token
prompt = "The quick brown fox"
token_ids = tokenizer.encode(prompt, return_tensors="pt")

print(f"Προτροπή: '{prompt}'")
print(f"ID των token: {token_ids}")
print(f"Μονάδες κειμένου (token): {[tokenizer.decode([t]) for t in token_ids[0]]}")

In [ ]:
# Πρόσθιο πέρασμα
with torch.no_grad():
    logits = model(token_ids)

print(f"Σχήμα των logits: {logits.shape}")

# Πάρε τις 5 πιθανότερες προβλέψεις για το επόμενο token
last_logits = logits[0, -1, :]
top_probs, top_indices = torch.softmax(last_logits, dim=-1).topk(5)

print(f"\nΟι 5 πιθανότερες προβλέψεις μετά το '{prompt}':")
for prob, idx in zip(top_probs, top_indices):
    token = tokenizer.decode([idx])
    print(f"  '{token}': {prob:.4f}")

print("\n(Τυχαίες προβλέψεις - το μοντέλο έχει τυχαία βάρη!)")

## 6. Φόρτωση προεκπαιδευμένων βαρών

Τώρα έρχεται το συναρπαστικό μέρος: ας φορτώσουμε τα πραγματικά βάρη του GPT-2 στο MiniGPT μας!

In [ ]:
def load_gpt2_weights(model, model_name="gpt2"):
    """
    Φόρτωσε τα προεκπαιδευμένα βάρη του GPT-2 στο μοντέλο μας MiniGPT.

    Ορίσματα:
        model: Το στιγμιότυπο του MiniGPT που χρησιμοποιούμε
        model_name: Όνομα μοντέλου στο HuggingFace ("gpt2", "gpt2-medium", κ.λπ.)

    Επιστρέφει:
        Το μοντέλο με τα βάρη φορτωμένα
    """
    from transformers import GPT2LMHeadModel

    print(f"Φόρτωση βαρών από το '{model_name}'...")

    # Φόρτωσε το μοντέλο από το HuggingFace
    hf_model = GPT2LMHeadModel.from_pretrained(model_name)
    hf_state = hf_model.state_dict()

    # Το λεξικό κατάστασης του μοντέλου μας
    our_state = model.state_dict()

    # Αντίγραψε τις διανυσματικές αναπαραστάσεις
    our_state['token_embed.weight'].copy_(hf_state['transformer.wte.weight'])
    our_state['pos_embed.weight'].copy_(hf_state['transformer.wpe.weight'])

    # Αντίγραψε κάθε μπλοκ Transformer
    for i in range(model.config.num_layers):
        # Κανονικοποιήσεις επιπέδου
        our_state[f'blocks.{i}.ln1.weight'].copy_(
            hf_state[f'transformer.h.{i}.ln_1.weight'])
        our_state[f'blocks.{i}.ln1.bias'].copy_(
            hf_state[f'transformer.h.{i}.ln_1.bias'])
        our_state[f'blocks.{i}.ln2.weight'].copy_(
            hf_state[f'transformer.h.{i}.ln_2.weight'])
        our_state[f'blocks.{i}.ln2.bias'].copy_(
            hf_state[f'transformer.h.{i}.ln_2.bias'])

        # Προσοχή: αντιμετάθεσε τις γραμμές και τις στήλες των πινάκων βαρών (.T). Αντίγραψε απευθείας τους όρους μετατόπισης.
        our_state[f'blocks.{i}.attn.qkv_proj.weight'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_attn.weight'].T)
        our_state[f'blocks.{i}.attn.qkv_proj.bias'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_attn.bias'])
        our_state[f'blocks.{i}.attn.out_proj.weight'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_proj.weight'].T)
        our_state[f'blocks.{i}.attn.out_proj.bias'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_proj.bias'])

        # Δίκτυο πρόσθιας τροφοδότησης: αντιμετάθεσε και εδώ τις γραμμές και τις στήλες των πινάκων βαρών (.T).
        our_state[f'blocks.{i}.ffn.fc1.weight'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_fc.weight'].T)
        our_state[f'blocks.{i}.ffn.fc1.bias'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_fc.bias'])
        our_state[f'blocks.{i}.ffn.fc2.weight'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_proj.weight'].T)
        our_state[f'blocks.{i}.ffn.fc2.bias'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_proj.bias'])

    # Τελική κανονικοποίηση επιπέδου
    our_state['ln_f.weight'].copy_(hf_state['transformer.ln_f.weight'])
    our_state['ln_f.bias'].copy_(hf_state['transformer.ln_f.bias'])

    # Σημείωση: Τα lm_head.weight και token_embed.weight χρησιμοποιούν κοινά βάρη, τα οποία έχουν ήδη αντιγραφεί

    print("Τα βάρη φορτώθηκαν με επιτυχία!")
    return model


In [ ]:
@torch.no_grad()
def generate_simple(model, tokenizer, prompt, max_new_tokens=20):
    """
    Παράγαγε κείμενο με άπληστη αποκωδικοποίηση (επίλεγε πάντα το πιθανότερο token).

    Ορίσματα:
        model: Το MiniGPT μας με φορτωμένα βάρη
        tokenizer: Ο tokenizer του GPT-2
        prompt: Αρχικό κείμενο
        max_new_tokens: Πόσα νέα token να παραχθούν

    Επιστρέφει:
        Συμβολοσειρά με το κείμενο που παράχθηκε
    """
    device = next(model.parameters()).device
    token_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # do_sample=False = άπληστη αποκωδικοποίηση: argmax σε κάθε βήμα
    output_ids = model.generate(
        token_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Αποκωδικοποίησε τα token ξανά σε κείμενο (μαζί με την προτροπή)
    return tokenizer.decode(output_ids[0])


In [ ]:
# Δημιούργησε το μοντέλο με τις ρυθμίσεις του GPT-2 Small
config = GPTConfig()  # Οι προεπιλεγμένες ρυθμίσεις αντιστοιχούν σε εκείνες του GPT-2 Small
model = MiniGPT(config)

# Φόρτωσε τα προεκπαιδευμένα βάρη
model = load_gpt2_weights(model, "gpt2")
model = model.to(device)

print(f"\nΣυσκευή του μοντέλου: {device}")
print(f"Παράμετροι: {count_parameters(model):,}")

In [ ]:
# ===== ΕΛΕΓΧΟΣ 6: Συμφωνία των logits με εκείνα του GPT-2 της HuggingFace =====
def test_gpt2_parity(model, tokenizer):
    """Έλεγξε ότι τα logits του μοντέλου που φορτώσαμε συμφωνούν με εκείνα του GPT-2 της HuggingFace."""
    from transformers import GPT2LMHeadModel

    hf_model = GPT2LMHeadModel.from_pretrained("gpt2").eval()
    model.eval()

    # Κάνε τη σύγκριση στη συσκευή του μοντέλου (το μοντέλο HF μεταφέρεται στην ίδια συσκευή)
    device = next(model.parameters()).device
    hf_model = hf_model.to(device)
    ids = tokenizer.encode("The quick brown fox jumps over",
                           return_tensors="pt").to(device)
    with torch.no_grad():
        our_logits = model(ids)
        hf_logits = hf_model(ids).logits

    # Οι πολύ μικρές διαφορές κινητής υποδιαστολής είναι αποδεκτές· μεγαλύτερη διαφορά σημαίνει ότι παραλείφθηκε κάποιο βάρος
    torch.testing.assert_close(our_logits, hf_logits, atol=1e-4, rtol=1e-4)
    print("Ο έλεγχος συμφωνίας με το GPT-2 ΠΕΤΥΧΕ!")

test_gpt2_parity(model, tokenizer)
# Αναμενόμενο: Ο έλεγχος συμφωνίας με το GPT-2 ΠΕΤΥΧΕ!


In [ ]:
# Παράγαγε κείμενο!
prompt = "The quick brown fox"
generated = generate_simple(model, tokenizer, prompt, max_new_tokens=30)

print(f"Προτροπή: '{prompt}'")
print(f"Παραγόμενο κείμενο: '{generated}'")

In [ ]:
# Δοκίμασε περισσότερες προτροπές!
prompts = [
    "Artificial intelligence will",
    "Once upon a time",
    "The capital of France is",
    "def fibonacci(n):"
]

for prompt in prompts:
    generated = generate_simple(model, tokenizer, prompt, max_new_tokens=20)
    print(f"'{prompt}' -> {generated}")
    print()

## 7. Σύγκριση τυχαίων και προεκπαιδευμένων βαρών

Σύγκρινε τα τυχαία βάρη με τα προεκπαιδευμένα για να δεις πόσο εντυπωσιακή είναι η διαφορά.

In [ ]:
# Μοντέλο με τυχαία βάρη
model_random = MiniGPT(GPTConfig())
model_random = model_random.to(device)

prompt = "Artificial intelligence will"

print("=" * 50)
print("ΤΥΧΑΙΑ ΒΑΡΗ:")
print(generate_simple(model_random, tokenizer, prompt, max_new_tokens=20))
print()
print("ΠΡΟΕΚΠΑΙΔΕΥΜΕΝΑ ΒΑΡΗ:")
print(generate_simple(model, tokenizer, prompt, max_new_tokens=20))
print("=" * 50)

print("\nΊδια αρχιτεκτονική. Ίδιος κώδικας. Η εκπαίδευση κάνει όλη τη διαφορά!")

## 8. Ασκήσεις

### Άσκηση 1: Μικροσκοπικό μοντέλο

Κατασκεύασε και δοκίμασε ένα μικροσκοπικό μοντέλο με συγκεκριμένες προδιαγραφές.

In [ ]:
# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ
# Κατασκεύασε ένα μικροσκοπικό MiniGPT με:
# - vocab_size=500
# - max_seq_len=16
# - embed_dim=64
# - num_heads=2
# - num_layers=2
# - d_ff=256

# 1. Όρισε τις ρυθμίσεις
# 2. Κατασκεύασε το μοντέλο
# 3. Εκτέλεσε ένα πρόσθιο πέρασμα με τυχαία token
# 4. Έλεγξε το σχήμα της εξόδου
# 5. Μέτρησε τις παραμέτρους

### Άσκηση 2: Πειραματισμός με τη θερμοκρασία

Τροποποίησε τη διαδικασία παραγωγής κειμένου ώστε να χρησιμοποιεί κλιμάκωση με βάση τη θερμοκρασία.

In [ ]:
# Ο ΚΩΔΙΚΑΣ ΣΟΥ ΕΔΩ
# Πρόσθεσε την παράμετρο temperature στη generate_simple. Στην κλήση της model.generate
# βάλε do_sample=True και πέρασε temperature=temperature. Η άπληστη επιλογή (argmax) αγνοεί τη θερμοκρασία.
# 
# Δοκίμασε τις θερμοκρασίες: 0.5, 1.0, 1.5
# Πώς αλλάζει η έξοδος;

### Άσκηση 3: Οπτικοποίηση της προσοχής

Οπτικοποίησε τα μοτίβα προσοχής στο προεκπαιδευμένο μοντέλο.

In [ ]:
# Ο ΚΩΔΙΚΑΣ ΣΟΥ ΕΔΩ
# 1. Εκτέλεσε το μοντέλο με return_attention=True
# 2. Πάρε τα βάρη προσοχής από το τελευταίο μπλοκ
# 3. Σχεδίασε έναν θερμικό χάρτη για την κεφαλή 0
# Υπόδειξη: Χρησιμοποίησε τη matplotlib.pyplot.imshow()

### Πρώτη ματιά στις ομάδες παραμέτρων (χρησιμοποιούνται στο Κεφάλαιο 12)

Στο Κεφάλαιο 12, η εξασθένηση βαρών εφαρμόζεται μόνο στις παραμέτρους που ωφελούνται από αυτήν. Αυτή η βοηθητική συνάρτηση χωρίζει τις παραμέτρους σε δύο ομάδες: στους πίνακες (2-D+) εφαρμόζεται εξασθένηση βαρών, ενώ στους όρους μετατόπισης και στα διανύσματα LayerNorm όχι.


In [ ]:
def get_parameter_groups(model, weight_decay=0.1):
    """
    Χώρισε τις παραμέτρους σε όσες πρέπει να υποβάλλονται σε εξασθένηση βαρών και σε όσες δεν πρέπει.

    Η εξασθένηση βαρών εφαρμόζεται στους πίνακες, συμπεριλαμβανομένων των πινάκων διανυσματικών αναπαραστάσεων,
    αλλά όχι στους όρους μετατόπισης 1-D ούτε στις παραμέτρους LayerNorm.
    Ορισμένες προσεγγίσεις εξαιρούν και τις διανυσματικές αναπαραστάσεις, αλλά εδώ εφαρμόζουμε παντού τον απλό κανόνα με βάση το πλήθος των διαστάσεων.
    """
    decay_params = []
    no_decay_params = []

    for name, param in model.named_parameters():
        # Στους πίνακες (2D+) εφαρμόζεται εξασθένηση βαρών, ενώ στα διανύσματα (όροι μετατόπισης, συντελεστές κλίμακας του LayerNorm) όχι
        if param.ndim < 2:
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    return [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0}
    ]

# Στο Κεφάλαιο 12, ο βελτιστοποιητής δημιουργείται με αυτή τη βοηθητική συνάρτηση:
# AdamW(get_parameter_groups(model, weight_decay=0.01), lr=learning_rate)
print("Ορίστηκε η get_parameter_groups!")


## Σύνοψη

**Τι φτιάξαμε:**

1. **GPTConfig**: Σαφές πρότυπο διαμόρφωσης
2. **MiniGPT**: Πλήρες γλωσσικό μοντέλο με κοινή χρήση βαρών
3. **Βασικοί έλεγχοι ορθότητας**: 6 δοκιμές για την επαλήθευση της ορθότητας, μεταξύ των οποίων και ο έλεγχος ότι τα logits συμφωνούν με εκείνα του GPT-2 της HuggingFace
4. **Φόρτωση βαρών**: Μεταφορά των βαρών του GPT-2
5. **Παραγωγή κειμένου**: Άπληστη αποκωδικοποίηση

**Βασικές έννοιες:**

- `nn.ModuleList` για τη στοίβαξη επιπέδων
- Στο GPT-2 Small η έξοδος χρησιμοποιεί τον πίνακα διανυσματικών αναπαραστάσεων των token, αντί για δεύτερο πίνακα με 50.257 × 768 = 38.597.376 παραμέτρους.
- Κεφαλή γλωσσικού μοντέλου: `(batch, seq, embed_dim)` → `(batch, seq, vocab_size)`
- `model.train()` έναντι `model.eval()`
- Τι να προσέξεις στην κοινή χρήση βαρών: το `'lm_head.weight'` ΕΜΦΑΝΙΖΕΤΑΙ στα κλειδιά που επιστρέφει η `state_dict().keys()`· τα κοινά βάρη μοιράζονται τον ίδιο χώρο στη μνήμη, οπότε η `torch.save` τα γράφει μόνο μία φορά και το αρχείο παραμένει μικρό

**Στη συνέχεια:** Στο Κεφάλαιο 12 θα μάθεις να εκπαιδεύεις αυτό το μοντέλο από την αρχή!